# Lab 05 - Silver Layer: Data Quality, Governance & PII (SOLUTION)

### *Clean, Enrich, Protect*

> **Note:** This is the **SOLUTION** version of Lab 05. All blanks have been filled in with the correct answers. Use this as a reference after completing the TODO version.

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Profile** data and identify quality issues before cleaning
2. **Apply** data cleaning transformations (null handling, range validation, type casting)
3. **Create** business metrics as calculated columns
4. **Enrich** data by joining with reference tables
5. **Implement** Unity Catalog governance (access controls)
6. **Handle** PII data with column masking and dynamic views
7. **Write** production-ready Silver Delta tables

---

## Silver Layer Pipeline

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border: 2px solid #1B3A4B; border-radius: 8px; padding: 20px; margin: 10px 0; font-family: monospace;">
<div style="text-align: center; font-weight: bold; font-size: 16px; margin-bottom: 15px; color: #1B3A4B;">SILVER LAYER TRANSFORMATION PIPELINE</div>
<div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; gap: 8px;">
  <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #137CBD;">Bronze</div>
    <div style="font-size: 11px; color: #1B1F24;">Raw Data</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FF3621;">Profile</div>
    <div style="font-size: 11px; color: #1B1F24;">Understand</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #00A972; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #00A972;">Clean</div>
    <div style="font-size: 11px; color: #1B1F24;">Validate</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #137CBD;">Enrich</div>
    <div style="font-size: 11px; color: #1B1F24;">Join & Calc</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FF3621;">Protect PII</div>
    <div style="font-size: 11px; color: #1B1F24;">Mask & Govern</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #1B3A4B; border: 2px solid #1B3A4B; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FFFFFF;">Silver</div>
    <div style="font-size: 11px; color: #F9F7F4;">Clean Data</div>
  </div>
</div>
</div>
""")

SILVER LAYER TRANSFORMATION PIPELINE 
 
 
 Bronze 
 Raw Data 
 
 → 
 
 Profile 
 Understand 
 
 → 
 
 Clean 
 Validate 
 
 → 
 
 Enrich 
 Join & Calc 
 
 → 
 
 Protect PII 
 Mask & Govern 
 
 → 
 
 Silver 
 Clean Data

---

## Section 1: Configuration

**OBJECTIVE:** Configure the environment and define paths for the Silver Layer pipeline.  
These settings must match the values you used in Lab 04 (Bronze Layer).

In [ ]:
# Import required libraries
from pyspark.sql.functions import (
    col, count, sum as spark_sum, when, avg, min as spark_min, max as spark_max,
    to_date, year, month, round as spark_round,
    current_timestamp, lit, concat, sha2, regexp_replace
)
from pyspark.sql.types import *

print("Libraries imported successfully!")

In [ ]:
# ============================================================
# CONFIGURATION (UC External Location handles ADLS auth)
# ============================================================

STORAGE_ACCOUNT = ""  # Your ADLS storage account name
CONTAINER = "data"  # Your ADLS container name
WS_KEY = ""  # Your folder prefix in ADLS (leave empty if not using per-user folders)

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{WS_KEY}" if WS_KEY else f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BRONZE_PATH = f"{BASE_PATH}/bronze/placements"
SILVER_PATH = f"{BASE_PATH}/silver/placements"
ENTITIES_REF_PATH = f"{BASE_PATH}/raw/entities"
BRONZE_DATABASE = "bronze"
SILVER_DATABASE = "silver"
TABLE_NAME = "placements"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DATABASE}")
bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()

print("=" * 60)
print("LAB 05 CONFIGURATION")
print("=" * 60)
print(f"Workspace Key: {WS_KEY}")
print(f"Bronze Path:   {BRONZE_PATH}")
print(f"Silver Path:   {SILVER_PATH}")
print(f"Bronze records: {bronze_count}")

---

## Section 2: Data Profiling

**WHY:** Before cleaning data, you must understand it. Data profiling reveals null values, outliers, invalid formats, and distribution anomalies. Without profiling, you are cleaning blindly -- you might miss critical issues or apply the wrong transformations.

```
+-------------------------------------------------------------------+
|                    DATA PROFILING CHECKLIST                        |
+-------------------------------------------------------------------+
|                                                                   |
|   1. Schema check    - Are data types correct?                    |
|   2. Null analysis   - Which columns have missing values?         |
|   3. Range analysis  - Are numeric values within valid bounds?    |
|   4. Distinct values - Are categorical columns consistent?        |
|                                                                   |
+-------------------------------------------------------------------+
```

### Exercise 2.1: Read Bronze Data and Check Schema

Read the Bronze table created in Lab 04 and inspect its structure.

In [ ]:
# Read Bronze data from Delta table
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# Display the schema
print("Bronze layer schema:")
bronze_df.printSchema()

print(f"\nTotal records in Bronze: {bronze_df.count()}")

# Preview the data
display(bronze_df.limit(5))

### Exercise 2.2: Null Value Analysis

Count null values in every critical column.  
**SOLUTION:** Use `isNull()` to complete the null checks.

In [ ]:
# Null value analysis across critical columns
# SOLUTION: isNull() used for all three blanks

null_counts = bronze_df.select(
    count("*").alias("total_records"),
    spark_sum(when(col("placement_id").isNull(), 1).otherwise(0)).alias("null_placement_id"),
    spark_sum(when(col("market_value").isNull(), 1).otherwise(0)).alias("null_market_value"),
    spark_sum(when(col("book_value").isNull(), 1).otherwise(0)).alias("null_book_value"),
    spark_sum(when(col("valuation_date").isNull(), 1).otherwise(0)).alias("null_valuation_date")
)

print("Null value analysis:")
display(null_counts)

### Exercise 2.3: Value Range Analysis

Examine min, max, and average for numeric columns to detect outliers.

In [ ]:
# Value range analysis for numeric columns
print("Value ranges for numeric columns:")
display(
    bronze_df.select(
        spark_min("market_value").alias("min_market_value"),
        spark_max("market_value").alias("max_market_value"),
        spark_round(avg("market_value"), 2).alias("avg_market_value"),
        spark_min("book_value").alias("min_book_value"),
        spark_max("book_value").alias("max_book_value"),
        spark_round(avg("book_value"), 2).alias("avg_book_value")
    )
)

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Data profiling is the first step of any Silver layer pipeline. It tells you <em>what</em> to clean, <em>where</em> the problems are, and <em>how much</em> data you will lose during cleaning. Never skip this step.</span>
</div>
""")

Key Takeaway: Data profiling is the first step of any Silver layer pipeline. It tells you what to clean, where the problems are, and how much data you will lose during cleaning. Never skip this step.

---

## Section 3: Data Cleaning

Now that we understand the data quality issues, we apply cleaning rules:

```
+-------------------------------------------------------------------+
|                    DATA QUALITY RULES                              |
+-------------------------------------------------------------------+
|                                                                   |
|   RULE 1: NOT NULL CHECKS                                         |
|   - placement_id    must not be null (primary key)                |
|   - market_value    must not be null (required for calculations)  |
|   - book_value      must not be null (required for calculations)  |
|   - valuation_date  must not be null (required for partitioning)  |
|                                                                   |
|   RULE 2: RANGE VALIDATION                                        |
|   - market_value   >= 0  (no negative valuations)                 |
|   - book_value     > 0   (avoid division by zero)                 |
|                                                                   |
+-------------------------------------------------------------------+
```

### Exercise 3.1: Filter Null Values

Remove records with null values in critical columns.  
**SOLUTION:** Use `isNotNull()` to keep only valid records.

In [ ]:
# Filter out records with null values in critical columns
# SOLUTION: isNotNull() used for all four blanks

clean_df = bronze_df.filter(
    (col("placement_id").isNotNull()) &
    (col("market_value").isNotNull()) &
    (col("book_value").isNotNull()) &
    (col("valuation_date").isNotNull())
)

original_count = bronze_df.count()
after_null_filter = clean_df.count()

print(f"Original records:       {original_count}")
print(f"After null filtering:   {after_null_filter}")
print(f"Records removed (nulls): {original_count - after_null_filter}")

### Exercise 3.2: Range Validation

Ensure numeric values are within acceptable business ranges.  
**SOLUTION:** Use `0` for both range thresholds (`market_value >= 0`, `book_value > 0`).

In [ ]:
# Apply range validation rules
# SOLUTION: market_value >= 0, book_value > 0

validated_df = clean_df.filter(
    (col("market_value") >= 0) &
    (col("book_value") > 0)
)

after_range_filter = validated_df.count()

# Data quality summary
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"Original Bronze records:      {original_count}")
print(f"After null filtering:         {after_null_filter}")
print(f"After range validation:       {after_range_filter}")
print(f"Total records removed:        {original_count - after_range_filter}")
print(f"Quality rate:                 {(after_range_filter / original_count) * 100:.2f}%")

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Data cleaning is about applying <em>business rules</em>, not arbitrary filters. Each rule should be documented and justified. Track how many records you lose at each step to detect upstream data issues early.</span>
</div>
""")

Key Takeaway: Data cleaning is about applying business rules , not arbitrary filters. Each rule should be documented and justified. Track how many records you lose at each step to detect upstream data issues early.

---

## Section 4: Type Casting and Date Handling

Raw data often arrives with incorrect types (e.g., dates as strings). We must cast them to proper types for downstream analytics and partitioning.

```
+-------------------------------------------------------------------+
|   BEFORE                              AFTER                       |
|   valuation_date   STRING   --->    valuation_date   DATE         |
|                                                                   |
|   EXTRACTED COMPONENTS:                                           |
|   valuation_year    INT     <---    year(valuation_date)          |
|   valuation_month   INT     <---    month(valuation_date)         |
+-------------------------------------------------------------------+
```

### Exercise 4.1: Cast Valuation Date

Convert `valuation_date` from string to proper date type.  
**SOLUTION:** Use `to_date` to cast to date.

In [ ]:
# Cast valuation_date from string to date type
# SOLUTION: to_date

typed_df = validated_df.withColumn(
    "valuation_date",
    to_date(col("valuation_date"))
)

# Verify the type change
print("Schema after type casting:")
typed_df.select("valuation_date").printSchema()

### Exercise 4.2: Extract Year and Month Components

Extract year and month from the valuation date for time-based analysis and partitioning.  
**SOLUTION:** Use `year` and `month` functions.

In [ ]:
# Extract valuation_year and valuation_month from valuation_date
# SOLUTION: year and month

dated_df = typed_df \
    .withColumn("valuation_year", year(col("valuation_date"))) \
    .withColumn("valuation_month", month(col("valuation_date")))

# Display sample results
print("Date components extracted:")
display(
    dated_df.select(
        "placement_id",
        "valuation_date",
        "valuation_year",
        "valuation_month"
    ).limit(5)
)

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Type casting ensures consistent data types across the pipeline. Extracting date components (year, month) is essential for partitioning, time-series analysis, and efficient querying.</span>
</div>
""")

Key Takeaway: Type casting ensures consistent data types across the pipeline. Extracting date components (year, month) is essential for partitioning, time-series analysis, and efficient querying.

---

## Section 5: Business Metrics (Calculated Columns)

The Silver layer is where raw data becomes *useful*. We derive business metrics that analysts and downstream Gold tables depend on.

```
+-------------------------------------------------------------------+
|                    CALCULATED BUSINESS METRICS                     |
+-------------------------------------------------------------------+
|                                                                   |
|   METRIC 1: UNREALIZED GAIN/LOSS                                  |
|   unrealized_gain_loss = market_value - book_value                |
|   * Positive = Unrealized GAIN                                    |
|   * Negative = Unrealized LOSS                                    |
|                                                                   |
|   METRIC 2: GAIN/LOSS PERCENTAGE                                  |
|   gain_loss_pct = (market_value - book_value) / book_value * 100  |
|   * Measures return as percentage of original investment          |
|                                                                   |
+-------------------------------------------------------------------+
```

### Exercise 5.1: Calculate Unrealized Gain/Loss

**SOLUTION:** Use `col("market_value") - col("book_value")`

In [ ]:
# Calculate unrealized gain/loss = market_value - book_value
# SOLUTION: col("market_value") - col("book_value")

metrics_df = dated_df.withColumn(
    "unrealized_gain_loss",
    col("market_value") - col("book_value")
)

# Preview the result
print("Unrealized Gain/Loss calculated:")
display(
    metrics_df.select(
        "placement_id",
        "market_value",
        "book_value",
        "unrealized_gain_loss"
    ).limit(5)
)

### Exercise 5.2: Calculate Gain/Loss Percentage

Compute the gain/loss as a percentage of book value, rounded to 2 decimal places.  
**SOLUTION:** Use `col("book_value")` and multiply by `100`.

In [ ]:
# Calculate gain/loss percentage
# SOLUTION: (market_value - book_value) / book_value * 100, rounded to 2 decimals

metrics_df = metrics_df.withColumn(
    "gain_loss_pct",
    spark_round(
        (col("market_value") - col("book_value")) / col("book_value") * 100,
        2
    )
)

# Preview the result
print("Gain/Loss Percentage calculated:")
display(
    metrics_df.select(
        "placement_id",
        "market_value",
        "book_value",
        "unrealized_gain_loss",
        "gain_loss_pct"
    ).limit(5)
)

In [ ]:
# Analyze gain/loss distribution: gainers vs losers vs neutral
print("Gain/Loss Distribution:")
display(
    metrics_df.select(
        count("*").alias("total_placements"),
        spark_sum(when(col("unrealized_gain_loss") > 0, 1).otherwise(0)).alias("gainers"),
        spark_sum(when(col("unrealized_gain_loss") < 0, 1).otherwise(0)).alias("losers"),
        spark_sum(when(col("unrealized_gain_loss") == 0, 1).otherwise(0)).alias("neutral"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct"),
        spark_round(spark_min("gain_loss_pct"), 2).alias("min_gain_loss_pct"),
        spark_round(spark_max("gain_loss_pct"), 2).alias("max_gain_loss_pct")
    )
)

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Calculated columns transform raw data into business-meaningful metrics. The Silver layer is where this value creation happens. Always validate your formulas (e.g., check for division by zero) before applying at scale.</span>
</div>
""")

Key Takeaway: Calculated columns transform raw data into business-meaningful metrics. The Silver layer is where this value creation happens. Always validate your formulas (e.g., check for division by zero) before applying at scale.

---

## Section 6: Data Enrichment (Join with Reference Data)

Enrichment adds context from reference (dimension) tables. Here we join placements with the entities reference table to add entity names and regions.

```
+-------------------------------------------------------------------+
|                     DATA ENRICHMENT JOIN                          |
+-------------------------------------------------------------------+
|                                                                   |
|   PLACEMENTS TABLE              ENTITIES REFERENCE                |
|   +-------------------+         +-------------------+             |
|   | placement_id     |         | entity_id        |             |
|   | axa_entity_id ---|----+    | entity_name      |             |
|   | asset_class      |    |    | region           |             |
|   | market_value     |    |    | country          |             |
|   | book_value       |    |    +-------------------+             |
|   | ...              |    |             ^                        |
|   +-------------------+    +------------|                        |
|                                         |                        |
|                            JOIN ON: axa_entity_id = entity_id    |
|                                                                   |
+-------------------------------------------------------------------+
```

In [ ]:
# Load entities reference data from ADLS
# If the reference file does not exist, create it inline as a fallback

try:
    entities_df = spark.read.format("json").load(ENTITIES_REF_PATH)
    print(f"Entities reference loaded from: {ENTITIES_REF_PATH}")
except Exception as e:
    print(f"Reference file not found, creating inline: {e}")
    entities_data = [
        ("AXA-FR-001", "AXA France", "Europe", "France"),
        ("AXA-DE-001", "AXA Germany", "Europe", "Germany"),
        ("AXA-UK-001", "AXA UK", "Europe", "United Kingdom"),
        ("AXA-US-001", "AXA US", "Americas", "United States"),
        ("AXA-JP-001", "AXA Japan", "Asia Pacific", "Japan"),
        ("AXA-HK-001", "AXA Hong Kong", "Asia Pacific", "Hong Kong"),
        ("AXA-SG-001", "AXA Singapore", "Asia Pacific", "Singapore"),
        ("AXA-BE-001", "AXA Belgium", "Europe", "Belgium"),
        ("AXA-IT-001", "AXA Italy", "Europe", "Italy"),
        ("AXA-ES-001", "AXA Spain", "Europe", "Spain")
    ]
    entities_schema = StructType([
        StructField("entity_id", StringType(), True),
        StructField("entity_name", StringType(), True),
        StructField("region", StringType(), True),
        StructField("country", StringType(), True)
    ])
    entities_df = spark.createDataFrame(entities_data, entities_schema)

print(f"Entities reference records: {entities_df.count()}")
display(entities_df)

### Exercise 6.1: Join Placements with Entities

Enrich the placements data by joining with the entities reference.  
**SOLUTION:** Use `entity_id` as the join column and `"left"` as the join type.

In [ ]:
# Join placements with entities reference
# SOLUTION: entity_id for join column, "left" for join type

enriched_df = metrics_df.join(
    entities_df,
    metrics_df.axa_entity_id == entities_df.entity_id,
    "left"
)

# Display enriched data
print("Enriched placements data:")
display(
    enriched_df.select(
        "placement_id",
        "axa_entity_id",
        "entity_name",
        "region",
        "asset_class",
        "market_value"
    ).limit(10)
)

In [ ]:
# Check for unmatched entities (placements with no matching reference)
unmatched = enriched_df.filter(col("entity_name").isNull())
unmatched_count = unmatched.count()

print(f"Unmatched entity records: {unmatched_count}")

if unmatched_count > 0:
    print("\nUnmatched entity IDs:")
    display(
        unmatched.select("axa_entity_id")
        .distinct()
        .limit(10)
    )

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Data enrichment via joins adds business context to raw data. Always use a <code>left</code> join when you want to keep all records, then check for unmatched rows. Unmatched records indicate reference data gaps that should be reported upstream.</span>
</div>
""")

Key Takeaway: Data enrichment via joins adds business context to raw data. Always use a left join when you want to keep all records, then check for unmatched rows. Unmatched records indicate reference data gaps that should be reported upstream.

---

## Section 7: Unity Catalog Governance & PII Handling

In production, not everyone should see all data. **PII** (Personally Identifiable Information) like names, emails, and IDs must be protected. **Unity Catalog** provides row-level security and column masking to protect sensitive data without creating separate copies.

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border: 2px solid #1B3A4B; border-radius: 8px; padding: 20px; margin: 10px 0; font-family: monospace;">
<div style="text-align: center; font-weight: bold; font-size: 16px; margin-bottom: 15px; color: #1B3A4B;">UNITY CATALOG GOVERNANCE MODEL</div>

<div style="margin-bottom: 15px;">
<strong style="color: #1B1F24;">3-Level Namespace:</strong>
<div style="display: flex; align-items: center; justify-content: center; gap: 8px; margin-top: 8px;">
  <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 6px; padding: 8px 16px; text-align: center;">
    <div style="font-weight: bold; color: #137CBD;">Catalog</div>
    <div style="font-size: 11px; color: #1B1F24;">e.g. prod</div>
  </div>
  <div style="font-size: 20px; color: #1B3A4B;">.</div>
  <div style="background: #FFFFFF; border: 2px solid #00A972; border-radius: 6px; padding: 8px 16px; text-align: center;">
    <div style="font-weight: bold; color: #00A972;">Schema</div>
    <div style="font-size: 11px; color: #1B1F24;">e.g. silver</div>
  </div>
  <div style="font-size: 20px; color: #1B3A4B;">.</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 6px; padding: 8px 16px; text-align: center;">
    <div style="font-weight: bold; color: #FF3621;">Table</div>
    <div style="font-size: 11px; color: #1B1F24;">e.g. placements</div>
  </div>
</div>
</div>

<div style="margin-bottom: 10px;">
<strong style="color: #1B1F24;">Row-Level Security:</strong> <span style="color: #1B1F24;">Different users see different rows based on their group/role.</span>
<div style="background: #FFFFFF; border: 1px solid #1B3A4B; border-radius: 4px; padding: 8px; margin-top: 6px; font-size: 12px;">
  <div style="color: #1B1F24;"><span style="color: #137CBD; font-weight: bold;">europe_team</span> &rarr; sees only WHERE region = 'Europe'</div>
  <div style="color: #1B1F24;"><span style="color: #00A972; font-weight: bold;">americas_team</span> &rarr; sees only WHERE region = 'Americas'</div>
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">global_admin</span> &rarr; sees all rows (no filter)</div>
</div>
</div>

<div>
<strong style="color: #1B1F24;">Column Masking:</strong> <span style="color: #1B1F24;">Sensitive columns show masked values for unauthorized users.</span>
<div style="background: #FFFFFF; border: 1px solid #1B3A4B; border-radius: 4px; padding: 8px; margin-top: 6px; font-size: 12px;">
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">trader_name</span> &rarr; SHA2 hash for auditors: <code>a3f2c8...</code></div>
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">trader_email</span> &rarr; regex mask: <code>****@company.com</code></div>
</div>
</div>
</div>
""")

UNITY CATALOG GOVERNANCE MODEL 

 
 3-Level Namespace: 
 
 
 Catalog 
 e.g. prod 
 
 . 
 
 Schema 
 e.g. silver 
 
 . 
 
 Table 
 e.g. placements 
 
 
 

 
 Row-Level Security: Different users see different rows based on their group/role. 
 
 europe_team → sees only WHERE region = 'Europe' 
 americas_team → sees only WHERE region = 'Americas' 
 global_admin → sees all rows (no filter) 
 
 

 
 Column Masking: Sensitive columns show masked values for unauthorized users. 
 
 trader_name → SHA2 hash for auditors: a3f2c8... 
 trader_email → regex mask: ****@company.com

### Exercise 7.1: Write Silver Table with Unity Catalog Registration

Select the final columns, write the Silver Delta table, and register it in the metastore.

In [ ]:
# Select final columns for Silver table
silver_df = enriched_df.select(
    col("placement_id"),
    col("axa_entity_id"),
    col("entity_name"),
    col("region"),
    col("asset_class"),
    col("instrument_id"),
    col("market_value"),
    col("book_value"),
    col("unrealized_gain_loss"),
    col("gain_loss_pct"),
    col("currency"),
    col("valuation_date"),
    col("valuation_year"),
    col("valuation_month"),
    current_timestamp().alias("_processing_timestamp")
)

# Write to Silver Delta table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("valuation_year") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PATH)

# Register table in metastore (Unity Catalog namespace)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_DATABASE}.{TABLE_NAME}
    USING DELTA LOCATION '{SILVER_PATH}'
""")

print(f"Silver table written to: {SILVER_PATH}")
print(f"Table registered: {SILVER_DATABASE}.{TABLE_NAME}")
print(f"Records: {spark.table(f'{SILVER_DATABASE}.{TABLE_NAME}').count()}")

### Exercise 7.2: PII Masking with SHA2 Hash

In this exercise we simulate PII columns and demonstrate how to create **masked views** that protect sensitive data. In production, these columns might contain real trader names and emails.

In [ ]:
# Simulate adding PII columns for demonstration
df_with_pii = spark.table(f"{SILVER_DATABASE}.{TABLE_NAME}") \
    .withColumn("trader_name", concat(lit("Trader_"), col("placement_id"))) \
    .withColumn("trader_email", concat(col("placement_id"), lit("@company.com")))

print("Sample data WITH PII (visible to authorized users only):")
display(
    df_with_pii.select(
        "placement_id", "trader_name", "trader_email", "market_value"
    ).limit(5)
)

In [ ]:
# Create a masked view - this is how you protect PII in production
# SHA2 hashes trader_name (irreversible), regex masks the email local part

spark.sql(f"""
    CREATE OR REPLACE VIEW {SILVER_DATABASE}.placements_masked AS
    SELECT 
        placement_id,
        axa_entity_id,
        entity_name,
        asset_class,
        market_value,
        book_value,
        -- PII columns are masked using SHA2 hash and regex
        SHA2(CONCAT('Trader_', placement_id), 256) AS trader_name_masked,
        REGEXP_REPLACE(CONCAT(placement_id, '@company.com'), '(^[^@]+)', '****') AS trader_email_masked,
        valuation_date
    FROM {SILVER_DATABASE}.{TABLE_NAME}
""")

print("Masked view created successfully.")
print(f"\nSample from {SILVER_DATABASE}.placements_masked:")
display(spark.sql(f"SELECT * FROM {SILVER_DATABASE}.placements_masked LIMIT 5"))

### Exercise 7.3: Row-Level Security (Demonstration)

In production, Unity Catalog row filters restrict data automatically per user/group.  
Here we demonstrate the concept by creating a filtered view.

In [ ]:
# In production, you would use Unity Catalog row filters:
# ALTER TABLE silver.placements SET ROW FILTER filter_by_region ON (region)
# For this lab, we demonstrate the concept with a filtered view:

spark.sql(f"""
    CREATE OR REPLACE VIEW {SILVER_DATABASE}.placements_europe AS
    SELECT * FROM {SILVER_DATABASE}.{TABLE_NAME}
    WHERE region = 'Europe'
""")

europe_count = spark.table(f"{SILVER_DATABASE}.placements_europe").count()
total_count = spark.table(f"{SILVER_DATABASE}.{TABLE_NAME}").count()

print(f"Europe-only view: {europe_count} records (out of {total_count} total)")
print("In production, Unity Catalog row filters restrict this automatically per user/group.")

### Exercise 7.4: Grant Permissions (Demonstration SQL)

In a Unity Catalog-enabled workspace, you control access with `GRANT` statements.  
The following SQL shows what production access control looks like.

In [ ]:
# Grant permissions demonstration
# In production, you would run these SQL statements:

grant_statements = """
-- Grant full analysts access to the Silver table:
-- GRANT SELECT ON TABLE silver.placements TO `data_analysts_group`;

-- Grant auditors access to the masked view only:
-- GRANT SELECT ON VIEW silver.placements_masked TO `external_auditors`;

-- Grant Europe team access to their regional view:
-- GRANT SELECT ON VIEW silver.placements_europe TO `europe_team`;

-- View current permissions (for instructor to demonstrate):
-- SHOW GRANTS ON TABLE silver.placements;
"""

print("UNITY CATALOG GRANT STATEMENTS (for production use):")
print(grant_statements)

# Verify governance objects exist
result = spark.sql("SELECT 'Governance setup demonstrated' AS status")
display(result)

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Unity Catalog governance is about giving the <em>right people</em> access to the <em>right data</em>. Use <strong>column masking</strong> (SHA2, regex) to protect PII without duplicating data. Use <strong>row-level security</strong> (row filters or filtered views) to restrict access by region, team, or role. Use <strong>GRANT</strong> statements to control who can read what.</span>
</div>
""")

Key Takeaway: Unity Catalog governance is about giving the right people access to the right data . Use column masking (SHA2, regex) to protect PII without duplicating data. Use row-level security (row filters or filtered views) to restrict access by region, team, or role. Use GRANT statements to control who can read what.

---

## Section 8: Verification

Validate the Silver layer data and review the transformation results.

In [ ]:
# Check Silver table record count and preview
silver_result = spark.table(f"{SILVER_DATABASE}.{TABLE_NAME}")
print(f"Total records in Silver table: {silver_result.count()}")
print("\nSilver table schema:")
silver_result.printSchema()
display(silver_result.limit(5))

In [ ]:
# Analyze by asset_class
print("Performance by Asset Class:")
display(
    spark.table(f"{SILVER_DATABASE}.{TABLE_NAME}")
    .groupBy("asset_class")
    .agg(
        count("*").alias("num_placements"),
        spark_round(spark_sum("market_value"), 2).alias("total_market_value"),
        spark_round(spark_sum("unrealized_gain_loss"), 2).alias("total_gain_loss"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
    )
    .orderBy("total_market_value", ascending=False)
)

In [ ]:
# Analyze by entity_name
print("Performance by Entity:")
display(
    spark.table(f"{SILVER_DATABASE}.{TABLE_NAME}")
    .groupBy("entity_name", "region")
    .agg(
        count("*").alias("num_placements"),
        spark_round(spark_sum("market_value"), 2).alias("total_market_value"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
    )
    .orderBy("total_market_value", ascending=False)
    .limit(10)
)

In [ ]:
# View Delta table history
print("Silver table history:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`"))

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Always verify your Silver layer output with aggregation queries and Delta history. This confirms that cleaning, enrichment, and governance steps were applied correctly before downstream Gold tables consume the data.</span>
</div>
""")

Key Takeaway: Always verify your Silver layer output with aggregation queries and Delta history. This confirms that cleaning, enrichment, and governance steps were applied correctly before downstream Gold tables consume the data.

---

## Section 9: Setting Up the Orchestration Pipeline

> *"In production, you don't run notebooks manually. Databricks Workflows let you orchestrate your pipeline -- Bronze, Silver, Gold -- with scheduling, monitoring, and alerting."*

Now that you have completed both the Bronze and Silver layers, let's create a **Databricks Workflow** that chains them together.

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Databricks Workflow: Medallion Pipeline</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 8px; flex-wrap: wrap; margin-top: 15px;">
    <div style="text-align: center;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 10px; padding: 14px 18px; min-width: 120px;">
        <div style="font-size: 11px; color: #757575; margin-bottom: 4px;">Task 1</div>
        <div style="font-weight: bold; color: #CD7F32;">bronze_ingestion</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">04_Bronze_Layer</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #FF3621;">&#8594;</div>
    <div style="text-align: center;">
      <div style="background: #F5F5F5; border: 3px solid #9E9E9E; border-radius: 10px; padding: 14px 18px; min-width: 120px; box-shadow: 0 0 12px rgba(158,158,158,0.3);">
        <div style="font-size: 11px; color: #757575; margin-bottom: 4px;">Task 2</div>
        <div style="font-weight: bold; color: #757575;">silver_transformation</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">05_Silver_Layer</div>
        <div style="font-size: 10px; background: #FF3621; color: #FFFFFF; border-radius: 4px; padding: 2px 8px; margin-top: 6px; font-weight: bold; display: inline-block;">YOU ARE HERE</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #757575;">&#8594;</div>
    <div style="text-align: center;">
      <div style="background: #FFFDE7; border: 2px dashed #F9A825; border-radius: 10px; padding: 14px 18px; min-width: 120px;">
        <div style="font-size: 11px; color: #757575; margin-bottom: 4px;">Task 3</div>
        <div style="font-weight: bold; color: #F9A825;">gold_aggregations</div>
        <div style="font-size: 11px; color: #757575; margin-top: 4px;">Lab 06</div>
      </div>
    </div>
  </div>
  <p style="text-align: center; color: #1B1F24; font-size: 12px; margin-top: 10px; margin-bottom: 0;">Task 3 (Gold) will be added in Lab 06</p>
</div>
""")

Databricks Workflow: Medallion Pipeline 
 
 
 
 Task 1 
 bronze_ingestion 
 04_Bronze_Layer 
 
 
 → 
 
 
 Task 2 
 silver_transformation 
 05_Silver_Layer 
 YOU ARE HERE 
 
 
 → 
 
 
 Task 3 
 gold_aggregations 
 Lab 06 
 
 
 
 Task 3 (Gold) will be added in Lab 06

### Creating the Databricks Workflow

Follow these steps to create your Medallion pipeline:

1. In the left sidebar, click **Jobs & Pipelines**
2. Click **Create Job**
3. Name the job: 
4. Add **Task 1** (Bronze):
   - Task name: 
   - Type: **Notebook**
   - Notebook path: 
5. Add **Task 2** (Silver):
   - Task name: 
   - Type: **Notebook**
   - Notebook path: 
   - Depends on: 
6. Configure compute (use serverless or the shared training cluster)
7. Click **Save** (we will add the Gold task in Lab 06)

In [ ]:
# Workflow configuration as JSON -- for reference only
import json

job_config = {
    "name": "Medallion_Pipeline",
    "tasks": [
        {
            "task_key": "bronze_ingestion",
            "description": "Ingest raw data into Bronze layer",
            "notebook_task": {
                "notebook_path": "/labs/jour2/04_Bronze_Layer"
            }
        },
        {
            "task_key": "silver_transformation",
            "depends_on": [{"task_key": "bronze_ingestion"}],
            "description": "Clean, enrich, and protect data in Silver layer",
            "notebook_task": {
                "notebook_path": "/labs/jour2/05_Silver_Layer"
            }
        }
    ],
    "max_concurrent_runs": 1,
    "format": "MULTI_TASK"
}

print("Workflow configuration (Bronze + Silver):")
print(json.dumps(job_config, indent=2))

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border-left: 4px solid #00A972; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong style="color: #1B3A4B;">Key Takeaway:</strong> <span style="color: #1B1F24;">Databricks Workflows orchestrate your pipeline by chaining notebook tasks with dependencies. Bronze runs first, then Silver runs only if Bronze succeeds. You will add the Gold task in Lab 06 to complete the pipeline.</span>
</div>
""")

Key Takeaway: Databricks Workflows orchestrate your pipeline by chaining notebook tasks with dependencies. Bronze runs first, then Silver runs only if Bronze succeeds. You will add the Gold task in Lab 06 to complete the pipeline.

---

## What You Learned

In [1]:
displayHTML("""
<table style="width: 100%; border-collapse: collapse; font-size: 14px;">
<thead>
  <tr style="background-color: #1B3A4B;">
    <th style="border: 1px solid #1B3A4B; padding: 10px; text-align: left; color: #FFFFFF;">Concept</th>
    <th style="border: 1px solid #1B3A4B; padding: 10px; text-align: left; color: #FFFFFF;">What You Learned</th>
  </tr>
</thead>
<tbody>
  <tr style="background-color: #FFFFFF;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Data Profiling</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Always understand your data before cleaning -- profile nulls, ranges, and distributions first</td>
  </tr>
  <tr style="background-color: #F9F7F4;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Null Handling</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Filter or fill nulls based on business rules; track how many records are removed at each step</td>
  </tr>
  <tr style="background-color: #FFFFFF;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Type Casting</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Ensure consistent data types (string to date, extract year/month) for downstream analytics</td>
  </tr>
  <tr style="background-color: #F9F7F4;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Business Metrics</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Derive value from raw data with calculated columns (gain/loss, percentages)</td>
  </tr>
  <tr style="background-color: #FFFFFF;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">PII Masking</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">SHA2 hash for irreversible masking; regex replacement for partial masking of emails</td>
  </tr>
  <tr style="background-color: #F9F7F4;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Row-Level Security</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Restrict data access by user/group using Unity Catalog row filters or filtered views</td>
  </tr>
  <tr style="background-color: #FFFFFF;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Dynamic Views</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Create masked/filtered views for different audience security levels without duplicating data</td>
  </tr>
  <tr style="background-color: #F9F7F4;">
    <td style="border: 1px solid #1B3A4B; padding: 10px; font-weight: bold; color: #1B3A4B;">Orchestration</td>
    <td style="border: 1px solid #1B3A4B; padding: 10px; color: #1B1F24;">Chain Bronze and Silver tasks with dependencies using Databricks Workflows</td>
  </tr>
</tbody>
</table>
""")

Concept,What You Learned
Data Profiling,"Always understand your data before cleaning -- profile nulls, ranges, and distributions first"
Null Handling,Filter or fill nulls based on business rules; track how many records are removed at each step
Type Casting,"Ensure consistent data types (string to date, extract year/month) for downstream analytics"
Business Metrics,"Derive value from raw data with calculated columns (gain/loss, percentages)"
PII Masking,SHA2 hash for irreversible masking; regex replacement for partial masking of emails
Row-Level Security,Restrict data access by user/group using Unity Catalog row filters or filtered views
Dynamic Views,Create masked/filtered views for different audience security levels without duplicating data
Orchestration,Chain Bronze and Silver tasks with dependencies using Databricks Workflows


<br>

**Next Steps:** Proceed to **Lab 06 - Gold Layer** to build business-level aggregations and analytics-ready datasets on top of this Silver table.